# 06 · MCP: desacoplando ferramentas do agente

Nos notebooks anteriores, cada ferramenta era uma função Python codificada no mesmo arquivo do
agente. Isso impede compartilhamento: para reutilizar uma ferramenta em outro agente, copia-se o
código; para usar a ferramenta de um terceiro, reescreve-se.

O **Model Context Protocol (MCP)**, proposto pela Anthropic em 2024, padroniza essa fronteira.
Define um protocolo uniforme pelo qual um agente (o *cliente*) **descobre e invoca** ferramentas
expostas por um *servidor*, seja ele local ou remoto, próprio ou de terceiros. O agente deixa de
conhecer as ferramentas em tempo de escrita e passa a descobri-las em tempo de execução.

O ponto conceitual: do ponto de vista do laço, uma ferramenta MCP é indistinguível de uma função
local. O que muda é a **origem** das ferramentas e quem as mantém — não a lógica do agente.

> **Execução.** Diferentemente dos notebooks 01–05, este requer o pacote `fastmcp` e usa
> `async`/`await` (suportado nativamente por células Jupyter/Colab). Por isso, ele **não acompanha
> um gabarito pré-executado**: execute-o em um ambiente Jupyter ou Colab. O LLM continua sendo
> servido exclusivamente pela NVIDIA.


## Dependências

In [ ]:
!pip install -q openai fastmcp

## Configuração do cliente (NVIDIA)

Idêntica à dos notebooks anteriores: o LLM é servido pela NVIDIA (no Colab, a chave pode vir de
*Secrets* com o nome `NVIDIA_API_KEY`). O MCP atua apenas na camada de ferramentas.


In [ ]:
import os
import json
from getpass import getpass
from openai import OpenAI

def _obter_api_key():
    try:                                  # 1) Colab Secrets (icone de chave a esquerda)
        from google.colab import userdata
        chave = userdata.get("NVIDIA_API_KEY")
        if chave:
            return chave
    except Exception:
        pass
    return os.environ.get("NVIDIA_API_KEY") or getpass("API key (nvapi-...): ")  # 2) env var ou 3) manual

BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL    = "meta/llama-3.1-8b-instruct"
API_KEY  = _obter_api_key()

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("Cliente configurado:", BASE_URL, "| modelo:", MODEL)


## O servidor MCP

Definimos um servidor com duas ferramentas. Em FastMCP, o *docstring* de cada função torna-se a
descrição exposta ao cliente, e a assinatura de tipos gera o esquema dos parâmetros
automaticamente. Em um cenário real, este servidor seria um processo separado — possivelmente em
outra máquina; aqui o instanciamos no próprio notebook e o conectamos *in-memory*, o que é
suficiente para evidenciar a fronteira de protocolo.


In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("mini-ferramentas")

@mcp.tool
def to_uppercase(text: str) -> str:
    "Converte um texto para letras maiusculas."
    return text.upper()

@mcp.tool
def contar_palavras(text: str) -> int:
    "Conta o numero de palavras em um texto."
    return len(text.split())


## Descoberta de ferramentas pelo cliente

O cliente conecta-se ao servidor e **lista** as ferramentas disponíveis. Note que o agente não as
declarou: ele as recebe do servidor em tempo de execução, já descritas e com seus esquemas.


In [ ]:
from fastmcp import Client

async with Client(mcp) as session:
    ferramentas_mcp = await session.list_tools()

for t in ferramentas_mcp:
    print(f"- {t.name}: {t.description}")


## Adaptação dos esquemas MCP para o formato da API

As ferramentas MCP trazem seus esquemas no formato JSON Schema padrão. A função de tradução abaixo
os converte para o formato de *function calling* esperado pela API — uma adaptação puramente
sintática.


In [ ]:
def mcp_para_openai(ferramentas_mcp):
    return [
        {
            "type": "function",
            "function": {
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema,
            },
        }
        for t in ferramentas_mcp
    ]


## O agente sobre ferramentas MCP

O laço é o mesmo dos notebooks anteriores, com duas diferenças: as ferramentas (`TOOLS`) são
**descobertas** do servidor, e a execução de cada invocação é feita por `session.call_tool` em vez
de uma chamada de função local. A função é `async` porque a sessão MCP é assíncrona. Mantemos a
salvaguarda de uma única invocação por turno (restrição do endpoint llama da NVIDIA).


In [ ]:
async def run_agent(task, model=MODEL, verbose=True):
    async with Client(mcp) as session:
        TOOLS = mcp_para_openai(await session.list_tools())

        messages = [
            {"role": "system", "content": "Use ferramentas quando precisar. Quando tiver a resposta final, responda sem chamar ferramentas."},
            {"role": "user", "content": task},
        ]
        while True:
            r = client.chat.completions.create(model=model, messages=messages, tools=TOOLS, tool_choice="auto")
            msg = r.choices[0].message
            if msg.tool_calls and len(msg.tool_calls) > 1:
                msg.tool_calls = msg.tool_calls[:1]
            messages.append(msg)
            if not msg.tool_calls:
                return msg.content

            tc = msg.tool_calls[0]
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"   -> {name}({args}) [via MCP]")

            res = await session.call_tool(name, args)
            # Extracao defensiva do conteudo retornado pelo servidor MCP
            try:
                texto = res.data if getattr(res, "data", None) is not None else res.content[0].text
            except Exception:
                texto = str(res)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(texto)})


## Execução

Duas tarefas independentes, cada uma exercitando uma das ferramentas **descobertas** do servidor
MCP. O agente não as conhecia em tempo de escrita; obteve-as do servidor em tempo de execução.


In [ ]:
print(await run_agent("Quantas palavras ha na frase 'agentes nao tem magica'?"))

In [ ]:
print(await run_agent("Converta para letras maiusculas: 'agentes nao tem magica'."))

## Por que isto importa

A fronteira que separa o agente do servidor é uma fronteira de **processo e protocolo**, não de
linguagem. O mesmo cliente que usamos aqui poderia conectar-se, sem alteração no código do agente,
a um servidor remoto mantido por terceiros — GitHub, bancos de dados, sistemas de arquivos — desde
que esse servidor fale MCP. É esse desacoplamento que permite ao ecossistema escalar: uma
ferramenta é escrita uma vez, como servidor, e reutilizada por qualquer agente compatível.

---
## Encerramento da sequência

- **01** — o agente como laço de controle.
- **02** — composição de ferramentas e o histórico como memória.
- **03** — o modelo como componente substituível e a dependência em *function calling*.
- **04** — robustez: isolamento de falhas, *retry*, terminação e conformidade com a plataforma.
- **05** — delegação hierárquica entre modelos.
- **06** — desacoplamento de ferramentas via MCP.

Cada camada acrescentada — memória, robustez, delegação, interoperabilidade — assenta sobre o
mesmo laço elementar do notebook 01. É essa a tese da sequência: frameworks de agentes não são
mágicos; são esse laço, instrumentado.
